# Phase 11 — PostgreSQL Prediction Logging

This notebook demonstrates the FraudShield SQLAlchemy persistence layer and investigation workflow. It will **not** silently create tables or insert fake production predictions. Configure `.env`, start PostgreSQL, and run `alembic upgrade head` first.

Data design: `prediction_records` stores model/policy version, timestamp, score, risk decision, latency, and a compact SHAP summary. `investigation_events` preserves status-change history. Raw transaction features are not persisted by default.

## 1. Start PostgreSQL and apply migrations

From PowerShell in the repository root:

```powershell
Copy-Item .env.example .env   # only if .env does not already exist; otherwise merge DATABASE_URL and DATABASE_REQUIRED
docker compose up -d postgres
python -m pip install -r requirements.txt
alembic upgrade head
uvicorn api.main:app --reload --host 127.0.0.1 --port 8000
```

The development database password is for local use only. Do not commit `.env`.

In [ ]:
from src.persistence.database import get_database_manager

manager = get_database_manager()
print({
    "database_enabled": manager.enabled,
    "database_required": manager.required,
    "database_reachable": manager.ping() if manager.enabled else False,
})

if not manager.enabled:
    print("Persistence is disabled. Set DATABASE_URL in .env, start PostgreSQL and run alembic upgrade head.")
elif not manager.ping():
    print("Database URL is configured, but PostgreSQL is not reachable. Check docker compose logs postgres.")
else:
    print("Database is reachable. API prediction logging can be enabled.")

## 2. Submit a prediction through the API

Start Uvicorn in a separate terminal first. This example reads the Phase 10 request JSON and sends it to the local API. The API returns `prediction_id` and `persistence_status`. When the database is enabled, the row is stored. When disabled, model inference still works but there is no persistent row.

In [ ]:
import json
from pathlib import Path
import httpx

request_path = Path("reports/api/example_request.json")
with request_path.open("r", encoding="utf-8") as f:
    request_payload = json.load(f)

try:
    response = httpx.post(
        "http://127.0.0.1:8000/predict?include_shap=false",
        json=request_payload, timeout=30.0,
    )
    print("HTTP status:", response.status_code)
    response.raise_for_status()
    prediction_response = response.json()
    {key: prediction_response.get(key) for key in (
        "request_id", "prediction_id", "created_at", "model_version",
        "risk_score_0_100", "risk_level", "decision", "latency_ms", "persistence_status"
    )}
except httpx.HTTPError as exc:
    print("Could not call the API. Confirm Uvicorn is running at 127.0.0.1:8000.", exc)

## 3. Query prediction history

The database repository uses a paginated query and supports status/risk filters. The API equivalent is `GET /predictions?limit=20&offset=0&investigation_status=NEW`.

In [ ]:
from src.persistence.repository import list_predictions

if manager.enabled and manager.ping():
    recent, total = list_predictions(manager, limit=20, offset=0)
    print(f"Total records: {total}")
    [
        {key: row[key] for key in (
            "prediction_id", "created_at", "model_version", "risk_score_0_100",
            "risk_level", "prediction_latency_ms", "investigation_status"
        )}
        for row in recent
    ]
else:
    print("Connect to PostgreSQL before querying prediction history.")

## 4. Update an investigation status

Use the API route `PATCH /predictions/{prediction_id}/investigation` with a JSON payload such as:

```json
{
  "status": "IN_REVIEW",
  "note": "Analyst review started"
}
```

Allowed values: `NEW`, `IN_REVIEW`, `CONFIRMED_FRAUD`, `FALSE_POSITIVE`, `CLOSED`. Each update appends an event so the old status is not lost. This API is not authenticated yet—keep it local.

In [ ]:
# Example call if prediction_response contains a saved prediction ID.
# Do not run this cell unless you want to change that record's investigation status.
if "prediction_response" in globals() and prediction_response.get("prediction_id"):
    update_response = httpx.patch(
        f"http://127.0.0.1:8000/predictions/{prediction_response['prediction_id']}/investigation",
        json={"status": "IN_REVIEW", "note": "Notebook review example"},
        timeout=10.0,
    )
    print(update_response.status_code, update_response.json())
else:
    print("No saved prediction ID available. Submit a prediction with persistence enabled first.")

## 5. Important engineering choices

- Timestamps are generated as UTC-aware datetimes and stored in `TIMESTAMPTZ` on PostgreSQL.
- Model version and policy version are stored with each prediction so later model changes do not rewrite history.
- `prediction_latency_ms` measures inference and optional SHAP work before the database write; database latency is intentionally separate.
- The full transaction input payload is not stored by default.
- `DATABASE_REQUIRED=true` (default when `DATABASE_URL` is present) makes failed writes return HTTP 503 rather than silently dropping prediction logs.
- Production deployment still needs authentication/authorization, rate limits, secrets management, migrations in CI/CD, backups, and retention controls.